In [ ]:
!pip install -q transformers torch fastapi uvicorn pyngrok keybert sentence-transformers nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/mental_health_project/final_emotion_model"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

emotion_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

print("Emotion model loaded.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Emotion model loaded.


In [ ]:
!pip install -U google-genai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 29.2 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.47.0
    Uninstalling google-auth-2.47.0:
      Successfully uninstalled google-auth-2.47.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.68.0
    Uninstalling google-genai-1.68.0:
      Successfully uninstalled google-genai-1.68.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 

In [ ]:
from google import genai

GEMINI_API_KEY ="YOUR_API_KEY"

client = genai.Client(
    api_key=GEMINI_API_KEY
)

In [ ]:
from google import genai

client = genai.Client(
    api_key="YOUR_API_KEY"

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Can you help me ,I am  in urgent need "
)

print(response.text)

In [ ]:
!pip install -q keybert sentence-transformers

In [ ]:
from keybert import KeyBERT

keyword_model = KeyBERT()

print("Keyword model loaded.")

In [ ]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model=emotion_model,
    tokenizer=tokenizer,
    top_k=None
)

In [ ]:
id2label = {
    0: "admiration",
    1: "amusement",
    2: "anger",
    3: "annoyance",
    4: "approval",
    5: "caring",
    6: "confusion",
    7: "curiosity",
    8: "desire",
    9: "disappointment",
    10: "disapproval",
    11: "disgust",
    12: "embarrassment",
    13: "excitement",
    14: "fear",
    15: "gratitude",
    16: "grief",
    17: "joy",
    18: "love",
    19: "nervousness",
    20: "optimism",
    21: "pride",
    22: "realization",
    23: "relief",
    24: "remorse",
    25: "sadness",
    26: "surprise",
    27: "neutral"
}

In [ ]:
positive_emotions = [
    "joy",
    "love",
    "gratitude",
    "optimism",
    "admiration",
    "approval",
    "caring",
    "excitement",
    "pride",
    "relief",
    "amusement"
]

negative_emotions = [
    "sadness",
    "anger",
    "fear",
    "grief",
    "remorse",
    "disappointment",
    "disapproval",
    "disgust",
    "nervousness",
    "annoyance",
    "embarrassment",
    "confusion"
]

In [ ]:
def get_mood_type(emotions):

    if not emotions:
        return "neutral"

    dominant_emotion = emotions[0]["emotion"]

    if dominant_emotion == "neutral":
        return "neutral"

    positive_score = 0
    negative_score = 0

    for item in emotions:

        emotion = item["emotion"]
        score = item["score"]

        if emotion in positive_emotions:
            positive_score += score

        elif emotion in negative_emotions:
            negative_score += score

    if positive_score > negative_score:
        return "positive"

    if negative_score > positive_score:
        return "negative"

    return "neutral"


In [ ]:
def analyze_emotion(text, threshold=0.10):

    prediction = emotion_classifier(text)

    formatted_predictions = []

    for item in prediction[0]:

        label_id = int(item["label"].split("_")[1])

        emotion = id2label[label_id]

        score = round(item["score"], 4)

        if score >= threshold:

            formatted_predictions.append({
                "emotion": emotion,
                "score": score,
                "percentage": round(score * 100, 2)
            })

    formatted_predictions = sorted(
        formatted_predictions,
        key=lambda x: x["score"],
        reverse=True
    )

    dominant_emotion = (
        formatted_predictions[0]["emotion"]
        if formatted_predictions
        else "neutral"
    )

    mood_type = get_mood_type(
        formatted_predictions
    )

    return {
        "text": text,
        "mood_type": mood_type,
        "dominant_emotion": dominant_emotion,
        "detected_emotions": formatted_predictions
    }

In [ ]:
def clean_keywords(keywords):

    banned_words = {

    "feel",
    "feeling",
    "felt",
    "really",
    "very",
    "quite",
    "actually",
    "basically",
    "just",
    "because",
    "also",
    "today",
    "yesterday",
    "tomorrow",
    "require",
    "requires",
    "required",
    "regular"
}

    cleaned = []

    for item in keywords:

        keyword = item["keyword"].lower().strip()

        words = [

            word

            for word in keyword.split()

            if word not in banned_words
        ]

        keyword = " ".join(words)

        if not keyword:
            continue

        cleaned.append({

            "keyword": keyword,

            "score": item["score"]
        })

    cleaned = sorted(

        cleaned,

        key=lambda x: x["score"],

        reverse=True
    )

    final_keywords = []

    for item in cleaned:

        keyword = item["keyword"]

        redundant = False

        for existing in final_keywords:

            existing_keyword = existing["keyword"]

            if (

                keyword in existing_keyword

                or existing_keyword in keyword

            ):

                redundant = True

                break

        if not redundant:

            final_keywords.append(item)

    return final_keywords

def extract_keywords(text, top_n=5):

    keywords = keyword_model.extract_keywords(

        text,

        keyphrase_ngram_range=(1, 1),

        stop_words="english",

        top_n=15,

        use_mmr=True,

        diversity=0.8
    )

    formatted_keywords = []

    for keyword, score in keywords:

        formatted_keywords.append({

            "keyword": keyword,

            "score": round(score, 4)
        })

    formatted_keywords = clean_keywords(
        formatted_keywords
    )

    return formatted_keywords[:top_n]

In [ ]:
trigger_categories = {

    "academic stress": [

        "exam",
        "exams",

        "assignment",
        "assignments",

        "study",
        "studies",

        "college",
        "university",

        "academic",

        "semester",

        "bad",

        "marks",

        "test",
        "tests",

        "quiz",
        "quizzes",

        "grade",
        "grades",

        "cgpa",

        "result",
        "results"
    ],

    "work stress": [

        "job",

        "office",

        "manager",

        "meeting",
        "meetings",

        "salary",

        "work pressure",

        "coworker",
        "coworkers",

        "client",
        "clients",

        "project",
        "projects",

        "deadline",
        "deadlines",

        "workload",

        "boss",

        "internship",

        "startup",

        "team",

        "promotion"
    ],

    "relationship issues": [

        "breakup",

        "relationship",
        "relationships",

        "partner",

        "girlfriend",

        "boyfriend",

        "family",

        "friend",
        "friends",

        "wife",

        "husband",

        "marriage",

        "parent",
        "parents",

        "mother",

        "father",

        "argument",
        "arguments"
    ],

    "financial stress": [

        "money",

        "rent",

        "loan",
        "loans",

        "debt",

        "expense",
        "expenses",

        "bill",
        "bills",

        "income",

        "salary",

        "fees",

        "tuition",

        "payment",
        "payments",

        "financial",

        "bank",

        "savings"
    ],

    "health anxiety": [

        "health",

        "disease",

        "hospital",

        "pain",

        "illness",

        "doctor",

        "medical",

        "medicine",

        "medication",

        "sick",

        "injury",

        "therapy",

        "mental health",

        "anxiety"
    ]
}

In [ ]:
def detect_trigger(text, keywords):

    detected_triggers = []

    combined_text = text.lower()

    keyword_texts = [
        item["keyword"].lower()
        for item in keywords
    ]

    for trigger, trigger_keywords in trigger_categories.items():

        for word in trigger_keywords:

            if word in combined_text:
                detected_triggers.append(trigger)

            for keyword in keyword_texts:

                if word in keyword:
                    detected_triggers.append(trigger)

    return list(set(detected_triggers))

In [ ]:
def generate_summary(analysis_result):

    mood = analysis_result["mood_type"]

    emotion = analysis_result["dominant_emotion"]

    triggers = analysis_result["triggers"]

    keywords = [
        item["keyword"]
        for item in analysis_result["keywords"][:3]
    ]

    summary_parts = []

    if mood == "negative":

        summary_parts.append(
            f"The journal entry suggests emotional distress characterized by {emotion}."
        )

    elif mood == "positive":

        summary_parts.append(
            f"The journal entry reflects a positive emotional state characterized by {emotion}."
        )

    else:

        summary_parts.append(
            f"The journal entry reflects a relatively neutral emotional state."
        )

    if triggers:

        summary_parts.append(
            f"Potential contributing factors include {', '.join(triggers)}."
        )

    if keywords:

        summary_parts.append(
            f"Key themes identified include {', '.join(keywords)}."
        )

    return " ".join(summary_parts)

In [ ]:
def generate_fallback_guidance(
    analysis_result
):

    emotion = analysis_result[
        "dominant_emotion"
    ]

    triggers = analysis_result[
        "triggers"
    ]

    mood_type = analysis_result[
        "mood_type"
    ]

    guidance_parts = []

    if mood_type == "negative":

        guidance_parts.append(
            f"The emotional patterns identified in this journal entry suggest "
            f"feelings associated with {emotion}. While challenging emotions "
            f"can be difficult to manage, recognizing them is often the first "
            f"step toward understanding the factors influencing emotional well-being."
        )

    elif mood_type == "positive":

        guidance_parts.append(
            "The overall tone of this journal entry reflects a positive emotional "
            "state. Identifying the situations and experiences that contribute "
            "to positive emotions can help reinforce healthy emotional patterns "
            "and improve long-term well-being."
        )

    else:

        guidance_parts.append(
            "The journal entry reflects a relatively balanced emotional state. "
            "Even when emotions are not strongly positive or negative, regular "
            "self-reflection can provide valuable insight into emotional trends "
            "and personal well-being."
        )

    if "academic stress" in triggers:

        guidance_parts.append(
            "Academic responsibilities appear to be contributing to emotional "
            "pressure. Managing multiple assignments, examinations, or study-related "
            "commitments simultaneously can sometimes feel overwhelming. Breaking "
            "larger tasks into smaller achievable goals and maintaining a structured "
            "study routine may help reduce stress while improving productivity."
        )

    if "work stress" in triggers:

        guidance_parts.append(
            "Work-related demands may also be influencing your emotional state. "
            "Periods of continuous responsibility and pressure can make it more "
            "difficult to maintain balance. Prioritizing important tasks, setting "
            "realistic expectations, and taking short breaks throughout the day "
            "may help create a more manageable workload."
        )

    if "relationship issues" in triggers:

        guidance_parts.append(
            "Personal relationships appear to be an important factor in this entry. "
            "Situations involving family members, friends, or partners can have a "
            "significant impact on emotional well-being. Giving yourself time to "
            "process these experiences and communicating openly with trusted "
            "individuals may provide additional support and perspective."
        )

    if "financial stress" in triggers:

        guidance_parts.append(
            "Financial concerns appear to be occupying a significant portion of "
            "your attention. Uncertainty related to expenses, bills, or financial "
            "responsibilities can naturally contribute to feelings of emotional "
            f"strain and {emotion}. Focusing on immediate priorities and creating "
            "a practical plan for managing current obligations may help improve "
            "confidence and reduce uncertainty."
        )

    if "health anxiety" in triggers:

        guidance_parts.append(
            "Health-related concerns are evident within this journal entry. "
            "When physical health becomes a source of uncertainty, it can often "
            "lead to increased worry and mental fatigue. Maintaining healthy "
            "routines, relying on accurate information, and seeking professional "
            "guidance when necessary may help reduce unnecessary concern."
        )

    if not triggers:

        guidance_parts.append(
            "Although no specific trigger category was strongly identified, "
            "continuing regular journaling can help reveal emotional patterns "
            "and improve self-awareness over time. Consistent reflection often "
            "provides useful insight into factors that influence daily well-being."
        )

    guidance_parts.append(
        "Small and consistent efforts are often more effective than major changes. "
        "Monitoring emotional patterns over time may help identify strategies that "
        "support emotional resilience and overall mental well-being."
    )

    return " ".join(
        guidance_parts
    )

In [ ]:
def generate_personalized_guidance(
    analysis_result
):

    try:

        prompt = f"""
        You are a supportive mental wellness assistant.

        Mood Type:
        {analysis_result['mood_type']}

        Dominant Emotion:
        {analysis_result['dominant_emotion']}

        Triggers:
        {analysis_result['triggers']}

        Keywords:
        {[k['keyword'] for k in analysis_result['keywords']]}

        Summary:
        {analysis_result['summary']}

        Provide personalized guidance.

        Use plain text only.
        Keep under 100 words.
        """

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return " ".join(
            response.text.split()
        )

    except Exception:

        return generate_fallback_guidance(
        analysis_result
    )

In [ ]:
def analyze_mental_health(text):

    emotion_result = analyze_emotion(text)

    keywords = extract_keywords(text)

    triggers = detect_trigger(
        text,
        keywords
    )

    analysis_result = {

        "text": text,

        "mood_type":
            emotion_result["mood_type"],

        "dominant_emotion":
            emotion_result["dominant_emotion"],

        "detected_emotions":
            emotion_result["detected_emotions"],

        "keywords":
            keywords,

        "triggers":
            triggers
    }

    analysis_result["summary"] = (
        generate_summary(
            analysis_result
        )
    )

    analysis_result["guidance"] = (
        generate_personalized_guidance(
            analysis_result
        )
    )

    return analysis_result

In [ ]:
result = analyze_mental_health(
    "I am worried about rent and bills.I am getting so frustrated because of my finances"
)

print(result["guidance"])

In [ ]:
from collections import Counter

In [ ]:
def generate_advanced_weekly_insight(
    weekly_analysis
):

    if not weekly_analysis:

        return (
            "No weekly journal entries were "
            "available for analysis."
        )

    negative_days = 0
    positive_days = 0

    dominant_emotions = []

    triggers = []

    for item in weekly_analysis:

        dominant_emotions.append(
            item["dominant_emotion"]
        )

        triggers.extend(
            item["triggers"]
        )

        if item["mood_type"] == "negative":
            negative_days += 1

        elif item["mood_type"] == "positive":
            positive_days += 1

    emotion_counter = Counter(
        dominant_emotions
    )

    trigger_counter = Counter(
        triggers
    )

    if emotion_counter:

      most_common_emotion = (
        emotion_counter.most_common(1)[0][0]
    )
    else:
        most_common_emotion = "neutral"

    summary_parts = []

    summary_parts.append(
        f"During the week, the user experienced "
        f"{negative_days} negative days and "
        f"{positive_days} positive days."
    )

    summary_parts.append(
    f"The dominant emotional trend throughout "
    f"the week was {most_common_emotion}."
    )
    if trigger_counter:

        trigger_list = [
            trigger
            for trigger, _
            in trigger_counter.most_common(3)
        ]

        summary_parts.append(
            f"Major contributing factors included "
            f"{', '.join(trigger_list)}."
        )

    summary_parts.append(
        "These patterns may help identify "
        "recurring emotional trends over time."
    )

    return " ".join(summary_parts)

In [ ]:
def generate_weekly_fallback_guidance(
    weekly_summary
):


    return (
        "Consider focusing on the most frequent sources of stress, "
        "maintaining a balanced routine, and continuing regular journaling. "
        "Tracking emotional patterns over time may help improve emotional well-being."
    )

In [ ]:
def generate_weekly_guidance(
    weekly_summary
):

    try:

        prompt = f"""
        You are a supportive mental wellness assistant.

        Based on the following weekly report:

        {weekly_summary}

        Provide a personalized weekly insight.

        Requirements:
        - Write as one natural paragraph
        - Do not use headings
        - Do not use bullet points
        - Do not use markdown
        - Keep under 100 words
        - Use plain text only
        """

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return " ".join(
            response.text.split()
        )

    except Exception:

        return generate_weekly_fallback_guidance(
            weekly_summary
        )

In [ ]:
weekly_entries = [

    "I am stressed because of exams and assignments.",

    "Project deadlines are exhausting.",

    "I had a breakup and feel lonely.",

    "I am worried about rent and bills.",

    "I am happy because I got an internship."
]

In [ ]:
weekly_results = []

for entry in weekly_entries:

    result = analyze_mental_health(
        entry
    )

    weekly_results.append(
        result
    )

weekly_summary = (
    generate_advanced_weekly_insight(
        weekly_results
    )
)

print(weekly_summary)

weekly_guidance = (
    generate_weekly_guidance(
        weekly_summary
    )
)

print("\n")
print("Weekly Guidance:")
print(weekly_guidance)

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

app = FastAPI()

class TextInput(BaseModel):
    text: str

class JournalEntry(BaseModel):
    day: str
    text: str

class WeeklyInput(BaseModel):
    entries: List[JournalEntry]

print("FastAPI initialized.")

In [ ]:
@app.get("/")
def home():
    return {
        "status": "running",
        "service": "Mental Health AI API"
    }


@app.post("/emotion")
def emotion_api(data: TextInput):
    return analyze_emotion(data.text)


@app.post("/keywords")
def keyword_api(data: TextInput):
    return {
        "keywords": extract_keywords(data.text)
    }


@app.post("/triggers")
def trigger_api(data: TextInput):

    keywords = extract_keywords(data.text)

    return {
        "triggers": detect_trigger(
            data.text,
            keywords
        )
    }

@app.post("/summary")
def summary_api(data: TextInput):

    result = analyze_mental_health(data.text)

    return {
        "summary": result["summary"]
    }


@app.post("/analyze")
def analyze_api(data: TextInput):

    return analyze_mental_health(data.text)

@app.post("/weekly-analysis")
def weekly_analysis_api(data: WeeklyInput):

    weekly_results = []

    for entry in data.entries:

        result = analyze_mental_health(
            entry.text
        )

        weekly_results.append({

            "day": entry.day,

            "dominant_emotion":
                result["dominant_emotion"],

            "mood_type":
                result["mood_type"],

            "triggers":
                result["triggers"],

            "summary":
                result["summary"]
        })

    weekly_summary = (
    generate_advanced_weekly_insight(
        weekly_results
    )
)

    weekly_guidance = (
    generate_weekly_guidance(
        weekly_summary
    )
)

    return {

    "daily_analysis":
        weekly_results,

    "weekly_summary":
        weekly_summary,

    "weekly_guidance":
        weekly_guidance
}

In [ ]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

thread = threading.Thread(
    target=run_server
)

thread.start()

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(
    "3EQvg4k2MW4Z0ltS4dNXgPE6FNl_7CtTFQHTRsVFb6i7G1WJo"
)

public_url = ngrok.connect(8000)

print(public_url)

In [ ]:
print(
  analyze_mental_health(
    "I am worried about rents and bills,i am so frustrated with my management of finances that its going down"
)
)